In [1]:
%load_ext autoreload
%autoreload 2

from plonk.pipe import PlonkPipeline
from PIL import Image
import torch
device = torch.device("cuda")

/home/charles/miniforge3/envs/plonk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
pipeline = PlonkPipeline("local_models/Default_DINO_Model").to(device)

Loading weights from local directory


Using cache found in /home/charles/.cache/torch/hub/facebookresearch_dinov2_main


In [14]:
image_path = "demo/examples/luca.jpg"
image = Image.open(image_path)

In [3]:
gps_coords = pipeline(image, batch_size=1024)

/home/charles/code/EECE5639_Final_Project_Fork/plonk/models/samplers/riemannian_flow_sampler.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=dtype):


In [4]:
latitude, longitude, likelihood = pipeline.compute_likelihood_grid(image, grid_resolution_deg=1, batch_size=4096, cfg=0)

Computing likelihood over a 181x361 grid (65341 points)...


Computing Likelihood Grid:   0%|          | 0/16 [00:00<?, ?it/s]

Computing Likelihood Grid:   6%|▋         | 1/16 [01:06<16:30, 66.06s/it]

Likelihood NFE: 1784


Computing Likelihood Grid:  12%|█▎        | 2/16 [02:14<15:48, 67.73s/it]

...

Likelihood NFE: 1838


Computing Likelihood Grid:  94%|█████████▍| 15/16 [17:54<01:10, 70.50s/it]

Likelihood NFE: 1784


Computing Likelihood Grid: 100%|██████████| 16/16 [18:58<00:00, 71.18s/it]

Likelihood NFE: 1766


In [7]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import Normalize

# Create a figure with a map projection
plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

# Add map features
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')

# Create a mesh grid for the contour plot
lon_mesh, lat_mesh = np.meshgrid(longitude, latitude)

# Normalize the likelihood values for better visualization
# Higher values should be more prominent
norm = Normalize(vmin=likelihood.min(), vmax=likelihood.max())

# Create a filled contour plot of the likelihood values
contour = ax.contourf(lon_mesh, lat_mesh, likelihood, 
                      transform=ccrs.PlateCarree(),
                      cmap='viridis', norm=norm, levels=50, alpha=0.7)

# Add a colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', pad=0.02)
cbar.set_label('Log Likelihood')

# Mark the predicted GPS coordinates
ax.scatter(gps_coords[:, 1], gps_coords[:, 0], 
           color='white', marker='.', s=10, 
           transform=ccrs.PlateCarree(), 
           label='Predicted Locations')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

# Add title and legend
plt.title('Geolocation Likelihood Map')
plt.legend()

# Show the plot
plt.tight_layout()
plt.show()


ImportError: /home/charles/miniforge3/envs/plonk/lib/python3.10/site-packages/pyproj/../../.././libtiff.so.6: undefined symbol: jpeg12_write_raw_data, version LIBJPEG_8.0

In [17]:
# Compute localizability
localizability = pipeline.compute_localizability(image)
print(f"Localizability: {localizability}")

Likelihood NFE: 362
Localizability: 0.350289523601532


In [55]:
image_path = "demo/examples/beacon_hill.png"
image_out = "beacon_hill"
image = Image.open(image_path)

FileNotFoundError: [Errno 2] No such file or directory: 'demo/examples/beacon_hill.png'

In [56]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation

_, traj = pipeline(image, batch_size=1024, return_trajectories=True)

# num_frames = 10
# frame_indices = np.linspace(0, len(traj) - 1, num_frames, dtype=int)
frame_indices = np.arange(len(traj))
num_frames = len(frame_indices)
frames = [traj[i] for i in frame_indices]

fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.OCEAN, facecolor="white")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
ax.set_global()

scatter = ax.scatter([], [], s=8, color="steelblue", alpha=0.5,
                     transform=ccrs.PlateCarree())
title = ax.set_title("")

def update(i):
    pts = frames[i]
    scatter.set_offsets(np.c_[pts[:, 1], pts[:, 0]])
    step = frame_indices[i]
    title.set_text(f"Step {step}/{len(traj)-1}  (t={1 - step/(len(traj)-1):.2f} → 0)")
    return scatter, title

ani = FuncAnimation(fig, update, frames=num_frames, interval=600, blit=False)
ani.save(f"diagrams/{image_out}_diffusion_trajectory.mp4", writer="ffmpeg", fps=12, dpi=150)
plt.close()
print("Saved to diffusion_trajectory.mp4")

/home/charles/code/EECE5639_Final_Project_Fork/plonk/models/samplers/riemannian_flow_sampler.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=dtype):


Saved to diffusion_trajectory.mp4
